# Contestability Score Exploration

Explore the distribution and meaning of the contestability score (normalised Shannon entropy)
across training articles in Supabase.

**Goal:** Understand the score distribution, identify which topics are most/least contested,
and determine what visualisations belong on the dashboard.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Fetch data from Supabase

In [2]:
client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])

# Paginate to get all rows (Supabase default limit is 1000)
PAGE_SIZE = 500
all_rows = []
page = 0
while True:
    start = page * PAGE_SIZE
    resp = (
        client.table("articles")
        .select("id, dominant_topic, topic_num, dominant_topic_weight, contestability_score, dataset_type, article_date, source, type")
        .not_.is_("contestability_score", "null")
        .range(start, start + PAGE_SIZE - 1)
        .execute()
    )
    all_rows.extend(resp.data)
    if len(resp.data) < PAGE_SIZE:
        break
    page += 1

df = pd.DataFrame(all_rows)
df["article_date"] = pd.to_datetime(df["article_date"], errors="coerce")
print(f"Loaded {len(df)} articles with contestability scores")
print(f"Dataset types: {df['dataset_type'].value_counts().to_dict()}")
df.head()

APIError: {'message': 'column articles.type does not exist', 'code': '42703', 'hint': None, 'details': None}

## 2. Overall distribution — what does the spread look like?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df["contestability_score"], bins=50, edgecolor="white", alpha=0.8, color="steelblue")
axes[0].set_xlabel("Contestability Score (normalised entropy)")
axes[0].set_ylabel("Number of articles")
axes[0].set_title("Distribution of Contestability Scores")
axes[0].axvline(0.3, color="green", linestyle="--", alpha=0.7, label="Low/Moderate boundary")
axes[0].axvline(0.5, color="orange", linestyle="--", alpha=0.7, label="Moderate/High boundary")
axes[0].axvline(0.7, color="red", linestyle="--", alpha=0.7, label="High/Very high boundary")
axes[0].legend(fontsize=8)

# Box plot
axes[1].boxplot(df["contestability_score"], vert=True)
axes[1].set_ylabel("Contestability Score")
axes[1].set_title("Score Spread (box plot)")

plt.tight_layout()
plt.show()

print(df["contestability_score"].describe())

## 3. Band breakdown — how many articles fall in each uncertainty band?

In [ ]:
bins = [0, 0.3, 0.5, 0.7, 1.0]
labels = ["Low (0-0.3)", "Moderate (0.3-0.5)", "High (0.5-0.7)", "Very High (0.7-1.0)"]
df["uncertainty_band"] = pd.cut(df["contestability_score"], bins=bins, labels=labels, include_lowest=True)

band_counts = df["uncertainty_band"].value_counts().sort_index()
band_pcts = (band_counts / len(df) * 100).round(1)

summary = pd.DataFrame({"Count": band_counts, "% of corpus": band_pcts})
print(summary)
print(f"\n{band_pcts.get('High (0.5-0.7)', 0) + band_pcts.get('Very High (0.7-1.0)', 0):.1f}% of articles have genuinely contested classifications")

## 4. Mean contestability by topic — which topics are most/least certain?

In [ ]:
topic_stats = (
    df.groupby("dominant_topic")["contestability_score"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ["green" if m < 0.3 else "orange" if m < 0.5 else "red" for m in topic_stats["mean"]]
ax.barh(topic_stats.index, topic_stats["mean"], color=colors, edgecolor="white")
ax.set_xlabel("Mean Contestability Score")
ax.set_title("Mean Contestability by Topic (green = confident, red = contested)")
ax.axvline(0.3, color="green", linestyle="--", alpha=0.5)
ax.axvline(0.5, color="orange", linestyle="--", alpha=0.5)
ax.axvline(0.7, color="red", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("\nTop 5 most contested topics:")
print(topic_stats.sort_values("mean", ascending=False).head())
print("\nTop 5 most confident topics:")
print(topic_stats.sort_values("mean", ascending=True).head())

## 5. High-contestability count per topic

In [ ]:
df["high_contest"] = df["contestability_score"] >= 0.5

high_by_topic = (
    df.groupby("dominant_topic")
    .agg(
        total=("id", "count"),
        high_contest=("high_contest", "sum"),
    )
)
high_by_topic["pct_contested"] = (high_by_topic["high_contest"] / high_by_topic["total"] * 100).round(1)
high_by_topic = high_by_topic.sort_values("pct_contested", ascending=False)

print(high_by_topic.to_string())

## 6. Contestability over time — does uncertainty change?

In [ ]:
df_dated = df.dropna(subset=["article_date"]).copy()
df_dated["month"] = df_dated["article_date"].dt.to_period("M").dt.to_timestamp()

monthly = df_dated.groupby("month")["contestability_score"].agg(["mean", "median", "count"])
monthly = monthly[monthly["count"] >= 5]  # only months with enough data

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly["mean"], marker="o", label="Mean contestability", color="steelblue")
ax.fill_between(monthly.index, monthly["mean"] - 0.05, monthly["mean"] + 0.05, alpha=0.2, color="steelblue")
ax.axhline(0.5, color="orange", linestyle="--", alpha=0.5, label="High uncertainty threshold")
ax.axvline(pd.Timestamp("2024-07-04"), color="red", linestyle="-", alpha=0.7, label="2024 Election")
ax.set_xlabel("Month")
ax.set_ylabel("Mean Contestability Score")
ax.set_title("Contestability Over Time")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Contestability by organisation type

In [ ]:
if "type" in df.columns and df["type"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    type_order = df.groupby("type")["contestability_score"].mean().sort_values().index
    sns.boxplot(data=df, x="type", y="contestability_score", order=type_order, ax=ax)
    ax.set_xlabel("Organisation Type")
    ax.set_ylabel("Contestability Score")
    ax.set_title("Contestability by Organisation Type")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("'type' column not available — skip this chart")

## 8. Sanity check — dominant topic weight vs contestability

Higher dominant topic weight should mean lower entropy (inverse relationship).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    df["dominant_topic_weight"],
    df["contestability_score"],
    alpha=0.3, s=10, color="steelblue"
)
ax.set_xlabel("Dominant Topic Weight")
ax.set_ylabel("Contestability Score (entropy)")
ax.set_title("Dominant Topic Weight vs Contestability")
plt.tight_layout()
plt.show()

corr = df["dominant_topic_weight"].corr(df["contestability_score"])
print(f"Pearson correlation: {corr:.3f}  (expected: strongly negative)")

## 9. Summary

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total articles with scores: {len(df)}")
print(f"Mean contestability: {df['contestability_score'].mean():.3f}")
print(f"Median contestability: {df['contestability_score'].median():.3f}")
print(f"Articles with high contestability (>0.5): {(df['contestability_score'] > 0.5).sum()} ({(df['contestability_score'] > 0.5).mean()*100:.1f}%)")
print(f"Articles with low contestability (<0.3): {(df['contestability_score'] < 0.3).sum()} ({(df['contestability_score'] < 0.3).mean()*100:.1f}%)")
print()
print("Most confident topic:", topic_stats["mean"].idxmin(), f"({topic_stats['mean'].min():.3f})")
print("Most contested topic:", topic_stats["mean"].idxmax(), f"({topic_stats['mean'].max():.3f})")